# BioVision — training the vehicle specialist on VehiDE

13,945 images, 36,081 instances, seven damage types, annotated to an insurance
company's claim standards. See `docs/DECISIONS.md` ADR-026 for why VehiDE and
not CarDD, and `scripts/inspect_vehide.py` for the counts this notebook assumes.


## 0. Reproducibility

Everything downstream depends on these constants. Changing any of them invalidates
the published metrics, so they are declared once, here, and recorded in the run.

In [ ]:
SEED = 20260311
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

MODEL = "yolo11s-seg.pt"   # small: 139 ms/image on CPU vs 483 for medium (measured)
IMGSZ = 640                # VehiDE averages 1.7M px, so 640 is a real downscale;
                           # 960 is the fallback if the scratch row disappoints
EPOCHS = 100
BATCH = 16

# The class order IS the contract with backend/src/biovision/models/specialists/
# vehicle_yolo.py::VEHIDE_CLASSES. The model emits integer ids; reordering this list
# silently relabels every prediction, so the loader below asserts they match.
#
# Alphabetical, because the order has to be a rule rather than a habit -- see
# DamageType. Seven classes, matching what VehiDE actually contains (ADR-026):
# CarDD's `crack` and `tire_flat` do not exist in it, and `torn`, `missing_part`
# and `punctured` are 30% of its instances.
CLASSES = [
    "dent",
    "glass_shatter",
    "lamp_broken",
    "missing_part",
    "punctured",
    "scratch",
    "torn",
]

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
print(f"seed={SEED}  split={TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}  classes={CLASSES}")

In [ ]:
!pip -q install ultralytics

import torch
from ultralytics import YOLO

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Obtain CarDD

CarDD is distributed on request by its authors. Obtain your own copy through their
process and mount it below.

**Do not commit it, and do not re-host it.** This notebook consumes a copy; it never
publishes one.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# VehiDE, as extracted from the Kaggle archive. Note the doubled directories --
# that is how the archive nests them, and assuming otherwise cost a run that
# reported "2324 images missing" before anyone looked at the tree.
VEHIDE_ROOT = Path("/content/drive/MyDrive/datasets/vehide")
TRAIN_IMAGES = VEHIDE_ROOT / "image" / "image"              # 11,621 images
VAL_IMAGES = VEHIDE_ROOT / "validation" / "validation"      # 2,324 images
TRAIN_ANNOS = VEHIDE_ROOT / "0Train_via_annos.json"
VAL_ANNOS = VEHIDE_ROOT / "0Val_via_annos.json"

for path in (TRAIN_IMAGES, VAL_IMAGES, TRAIN_ANNOS, VAL_ANNOS):
    assert path.exists(), f"not found: {path}"

WORK = Path("/content/biovision")
WORK.mkdir(exist_ok=True)
print("VehiDE found:")
print(f"  train images: {len(list(TRAIN_IMAGES.glob('*.jpg'))):,}")
print(f"  val images:   {len(list(VAL_IMAGES.glob('*.jpg'))):,}")


## 2. Convert COCO annotations to YOLO segmentation format

CarDD ships COCO-style polygons. YOLO wants one `.txt` per image with normalised
polygon coordinates.

The class mapping is asserted rather than assumed: if CarDD's category names do not
match `CLASSES` after normalisation, the conversion stops. A silent mismatch here
would train a model whose class 1 means something other than what the API says it
means, and nothing downstream would look wrong.

In [ ]:
import json
from collections import defaultdict


def normalise(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def convert_coco(annotation_file: Path, image_dir: Path, out_dir: Path) -> int:
    """COCO instance segmentation -> YOLO segmentation labels.

    Rehearsed against synthetic COCO before the real dataset arrived. Two
    behaviours below come directly from that rehearsal, and both are about
    refusing to lose data quietly:

    * COCO stores masks either as polygons or as RLE. This converter only
      understands polygons; fed RLE it used to die with
      ``TypeError: unsupported operand type(s) for /: 'str' and 'int'`` --
      accurate, and useless at 3am in the middle of 4,000 images.
    * Images with no usable polygon, and annotations whose file is missing,
      are skipped. Skipping is right; skipping silently is not. A converter
      that turns 4,000 images into 3,100 without saying so would poison every
      number downstream, and nothing later in the pipeline could detect it.
    """
    coco = json.loads(annotation_file.read_text())

    categories = {c["id"]: normalise(c["name"]) for c in coco["categories"]}
    found = sorted(set(categories.values()))
    assert found == sorted(CLASSES), (
        f"CarDD categories {found} do not match CLASSES {sorted(CLASSES)}. "
        "Fix the mapping before training -- a mismatch relabels every prediction."
    )
    class_index = {name: CLASSES.index(name) for name in CLASSES}

    images = {img["id"]: img for img in coco["images"]}
    per_image = defaultdict(list)
    crowd = 0
    for ann in coco["annotations"]:
        if ann.get("iscrowd"):
            crowd += 1
            continue
        segmentation = ann.get("segmentation")
        if isinstance(segmentation, dict):
            raise RuntimeError(
                f"annotation {ann.get('id')} stores its mask as RLE, not polygons. "
                "This converter only reads polygons. Decode the RLE first "
                "(pycocotools.mask.decode -> cv2.findContours) or export the "
                "annotations in polygon form; do not let it fall through, or the "
                "instance is dropped and the class silently loses examples."
            )
        per_image[ann["image_id"]].append(ann)

    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    written = 0
    no_polygon: list[str] = []
    missing_file: list[str] = []
    degenerate = 0

    for image_id, annotations in per_image.items():
        info = images[image_id]
        width, height = info["width"], info["height"]
        lines = []

        for ann in annotations:
            for polygon in ann.get("segmentation", []):
                if len(polygon) < 6:  # fewer than three points is not a polygon
                    degenerate += 1
                    continue
                coords = []
                for i in range(0, len(polygon), 2):
                    coords.append(min(1.0, max(0.0, polygon[i] / width)))
                    coords.append(min(1.0, max(0.0, polygon[i + 1] / height)))
                index = class_index[categories[ann["category_id"]]]
                lines.append(str(index) + " " + " ".join(f"{c:.6f}" for c in coords))

        if not lines:
            no_polygon.append(info["file_name"])
            continue

        source = image_dir / info["file_name"]
        if not source.is_file():
            missing_file.append(info["file_name"])
            continue

        (out_dir / "labels" / f"{source.stem}.txt").write_text("\n".join(lines))
        (out_dir / "images" / source.name).write_bytes(source.read_bytes())
        written += 1

    # Say out loud what did not make it. An unexplained drop here is the kind of
    # thing that shows up much later as a class with suspiciously few examples.
    total = len(images)
    print(f"converted {written}/{total} images")
    if crowd:
        print(f"  {crowd} iscrowd annotation(s) skipped (COCO convention)")
    if degenerate:
        print(f"  {degenerate} polygon(s) with fewer than 3 points skipped")
    for label, names in (("no usable polygon", no_polygon), ("image file missing", missing_file)):
        if names:
            print(f"  {len(names)} skipped -- {label}: {', '.join(names[:5])}"
                  + (" ..." if len(names) > 5 else ""))
    dropped = total - written
    if dropped > total * 0.05:
        print(f"  WARNING: {dropped / total:.0%} of images were dropped. Investigate "
              "before training -- this is large enough to change the class balance.")

    return written


print("Point these at your CarDD layout, then run.")


## 2b. VehiDE: convert VIA annotations

VehiDE ships VGG Image Annotator (VIA) JSON, not COCO: a mapping of filename to
`regions`, each region carrying a shape and a class attribute. Same destination as
the COCO converter above, different source format.

The class attribute key is not fixed across VIA exports, so it is discovered rather
than assumed, and the discovered class names are printed for you to check against
`CLASSES` before anything is written.


In [ ]:
import json
from collections import Counter


def _region_class(region: dict) -> str | None:
    """Pull the class name out of a VIA region.

    Two shapes are accepted, because VehiDE does not write standard VIA:

    * VehiDE:  {"all_x": [...], "all_y": [...], "class": "tray_son"}
    * VIA:     {"shape_attributes": {...}, "region_attributes": {"damage": "..."}}

    Reading VehiDE as standard VIA parses cleanly and finds nothing -- the first
    inspection run reported 36,081 regions and zero classes before this was
    fixed. A rehearsal against synthetic *standard* VIA had passed, which is
    exactly the trap: the rehearsal proved the code ran, not that the assumption
    about the format held.
    """
    if isinstance(region.get("class"), str):
        return normalise(region["class"])

    attributes = region.get("region_attributes") or {}
    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalise(value)
    return None


def _region_points(region: dict) -> tuple[list, list]:
    """Polygon points, from either annotation shape."""
    if "all_x" in region:
        return list(region.get("all_x") or []), list(region.get("all_y") or [])
    shape = region.get("shape_attributes") or {}
    return list(shape.get("all_points_x") or []), list(shape.get("all_points_y") or [])


def _is_polygon(region: dict) -> bool:
    if "all_x" in region:
        return True
    return (region.get("shape_attributes") or {}).get("name") == "polygon"


def inspect_via(annotation_file: Path) -> None:
    """Print what is actually in the file, before converting anything.

    VehiDE's class names will not match CLASSES exactly -- it has eight types,
    including `torn` and `lost parts`, which CarDD does not. Read this output and
    write VIA_TO_CLASS below accordingly; do not skip it. A silent mismatch here
    relabels every prediction.
    """
    via = json.loads(annotation_file.read_text())
    attribute_keys: Counter = Counter()
    class_names: Counter = Counter()
    shapes: Counter = Counter()

    for entry in via.values():
        for region in entry.get("regions", []):
            attribute_keys.update(k for k in region if k not in {"all_x", "all_y"})
            name = _region_class(region)
            if name:
                class_names[name] += 1
            shapes["polygon" if _is_polygon(region) else "other"] += 1

    print(f"images in file:       {len(via)}")
    print(f"region attribute keys: {dict(attribute_keys)}")
    print(f"shape types:           {dict(shapes)}")
    print()
    print("class -> instance count")
    for name, count in class_names.most_common():
        marker = "  <-- in CLASSES" if name in CLASSES else ""
        print(f"  {name:24s} {count:6d}{marker}")


#: VehiDE's class names are Vietnamese. Each is given with its literal meaning and
#: the English term from the paper's own class list, so the mapping can be checked
#: rather than trusted. Run inspect_via and confirm these are the names present
#: before training -- a wrong entry here relabels every prediction of that class.
#:
#: Counts are from data/vehide as measured by scripts/inspect_vehide.py.
VIA_TO_CLASS: dict[str, str | None] = {
    "tray_son": "scratch",         # tray son     -- paint scratch   14,647  40.6%
    "mop_lom": "dent",             # mop lom      -- dent             5,681  15.7%
    "rach": "torn",                # rach         -- torn             5,509  15.3%
    "mat_bo_phan": "missing_part",  # mat bo phan  -- lost part        2,818   7.8%
    "be_den": "lamp_broken",       # be den       -- broken lights    2,782   7.7%
    "thung": "punctured",          # thung        -- punctured        2,423   6.7%
    "vo_kinh": "glass_shatter",    # vo kinh      -- broken glass     2,221   6.2%
}


def convert_via(annotation_file: Path, image_dir: Path, out_dir: Path) -> int:
    """VIA polygons -> YOLO segmentation labels.

    Mirrors convert_coco: same output layout, same refusal to lose data quietly.
    """
    if not VIA_TO_CLASS:
        raise RuntimeError(
            "VIA_TO_CLASS is empty. Run inspect_via first and write the mapping "
            "explicitly -- an implicit one would relabel instances silently."
        )
    unknown = {k for k in VIA_TO_CLASS.values() if k is not None and k not in CLASSES}
    if unknown:
        raise RuntimeError(f"VIA_TO_CLASS maps to names not in CLASSES: {sorted(unknown)}")

    via = json.loads(annotation_file.read_text())
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    from PIL import Image

    written = 0
    unmapped: Counter = Counter()
    non_polygon: Counter = Counter()
    no_regions: list[str] = []
    missing_file: list[str] = []
    dropped_by_choice = 0

    for entry in via.values():
        filename = entry.get("name") or entry.get("filename")
        if not filename:
            continue
        source = image_dir / filename
        if not source.is_file():
            missing_file.append(str(filename))
            continue

        with Image.open(source) as image:
            width, height = image.size

        lines = []
        for region in entry.get("regions", []):
            if not _is_polygon(region):
                # VIA also writes rect/circle/ellipse. A box is not a mask, and
                # area_ratio is computed from the mask -- see vehicle_yolo.py.
                shape_name = (region.get("shape_attributes") or {}).get("name", "?")
                non_polygon[shape_name] += 1
                continue

            raw = _region_class(region)
            if raw not in VIA_TO_CLASS:
                unmapped[raw] += 1
                continue
            target = VIA_TO_CLASS[raw]
            if target is None:
                dropped_by_choice += 1
                continue

            xs, ys = _region_points(region)
            if len(xs) < 3 or len(xs) != len(ys):
                continue

            coords = []
            for x, y in zip(xs, ys):
                coords.append(min(1.0, max(0.0, x / width)))
                coords.append(min(1.0, max(0.0, y / height)))
            lines.append(
                str(CLASSES.index(target)) + " " + " ".join(f"{c:.6f}" for c in coords)
            )

        if not lines:
            no_regions.append(str(filename))
            continue

        (out_dir / "labels" / f"{source.stem}.txt").write_text("\n".join(lines))
        (out_dir / "images" / source.name).write_bytes(source.read_bytes())
        written += 1

    total = len(via)
    print(f"converted {written}/{total} images")
    if dropped_by_choice:
        print(f"  {dropped_by_choice} instance(s) dropped by VIA_TO_CLASS (mapped to None)")
    if unmapped:
        print(f"  UNMAPPED classes -- these instances were lost: {dict(unmapped)}")
    if non_polygon:
        print(f"  non-polygon shapes skipped: {dict(non_polygon)}")
    for label, names in (("no usable polygon", no_regions), ("image missing", missing_file)):
        if names:
            print(f"  {len(names)} skipped -- {label}: {', '.join(names[:5])}"
                  + (" ..." if len(names) > 5 else ""))
    if total and (total - written) > total * 0.05:
        print(f"  WARNING: {(total - written) / total:.0%} of images dropped. Investigate "
              "before training -- large enough to change the class balance.")

    return written


print("Run inspect_via first, fill in VIA_TO_CLASS, then convert_via.")


## 3. Split — deterministic, and pinned

This is the cell that makes the reported metrics mean something. The split is a
function of the seed and the sorted filenames, so anyone with their own CarDD copy
reproduces exactly these three sets.

In [ ]:
import hashlib
import shutil


def split_dataset(converted: Path, target: Path) -> dict[str, int]:
    stems = sorted(p.stem for p in (converted / "images").iterdir())
    rng = random.Random(SEED)
    rng.shuffle(stems)

    n = len(stems)
    train_end = int(n * TRAIN_RATIO)
    val_end = train_end + int(n * VAL_RATIO)
    splits = {
        "train": stems[:train_end],
        "val": stems[train_end:val_end],
        "test": stems[val_end:],
    }

    for name, members in splits.items():
        for kind in ("images", "labels"):
            (target / name / kind).mkdir(parents=True, exist_ok=True)
        for stem in members:
            for source in (converted / "images").glob(f"{stem}.*"):
                shutil.copy2(source, target / name / "images" / source.name)
            label = converted / "labels" / f"{stem}.txt"
            if label.is_file():
                shutil.copy2(label, target / name / "labels" / label.name)

    # A fingerprint of the split itself. Record it with the metrics: if it ever
    # differs, the numbers are not comparable to the published ones.
    digest = hashlib.sha256(
        "|".join(f"{k}:{','.join(v)}" for k, v in sorted(splits.items())).encode()
    ).hexdigest()[:16]
    print(f"split fingerprint: {digest}")

    return {name: len(members) for name, members in splits.items()}


print("Run after conversion.")

In [ ]:
DATASET = WORK / "cardd_yolo"

data_yaml = WORK / "cardd.yaml"
data_yaml.write_text(
    f"""path: {DATASET}
train: train/images
val: val/images
test: test/images

# Order matters: it is the contract with vehicle_yolo.py::CARDD_CLASSES.
names:
"""
    + "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASSES))
)
print(data_yaml.read_text())

## 4. Train

Validation runs on the `val` split. The `test` split is not touched until section 5.

In [ ]:
model = YOLO(MODEL)

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    deterministic=True,
    patience=20,
    project=str(WORK / "runs"),
    name="cardd_seg",
    # Damage is small and often low-contrast; heavy colour jitter hurts more than
    # it helps. Keep the defaults conservative and let the report say what happened.
    hsv_v=0.3,
    degrees=5.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)

## 5. Evaluate on the held-out test split

Per class, deliberately. The literature finds `dent`, `scratch` and `crack` to be
the hard classes; if these numbers show the same, that is a correct result reported
honestly, not a defect to average away.

In [ ]:
best = WORK / "runs" / "cardd_seg" / "weights" / "best.pt"
trained = YOLO(str(best))

metrics = trained.val(data=str(data_yaml), split="test", imgsz=IMGSZ)

print("| Class | mAP@50 | mAP@50-95 | Precision | Recall |")
print("|---|---|---|---|---|")
for i, name in enumerate(CLASSES):
    p, r, m50, m5095 = metrics.seg.class_result(i)
    print(f"| {name} | {m50:.3f} | {m5095:.3f} | {p:.3f} | {r:.3f} |")
print(
    f"| **all** | {metrics.seg.map50:.3f} | {metrics.seg.map:.3f} | "
    f"{metrics.seg.mp:.3f} | {metrics.seg.mr:.3f} |"
)

## 6. Export

Copy `best.pt` to `backend/weights/cardd_yolo_seg.pt`, then:

1. publish it as a GitHub release artifact (the checkpoint, never the dataset);
2. record its SHA-256 in `backend/scripts/fetch_weights.py`;
3. paste the table above into README section 7.3 **verbatim**, weak rows included;
4. record the split fingerprint alongside it;
5. regenerate the golden set: `uv run python -m scripts.update_golden --review`.

In [ ]:
import hashlib

digest = hashlib.sha256(best.read_bytes()).hexdigest()
print(f"cardd_yolo_seg.pt\nsha256: {digest}\nsize:   {best.stat().st_size / 1e6:.1f} MB")